# **Ejercicio 9: Reflexion sobre accesibilidad y sesgo**

Discutir que limitaciones tiene este dataset (tono de piel, angulo de camara, iluminacion,
tamaño de mano) para un producto real de accesibilidad, y que haria falta para llevar este
prototipo a un caso de uso real como el de SignBridge.

El rubro pide **al menos una recomendacion concreta**.

## **Lo que ya se observo, no lo que se especula**

Esta discusion no parte de cero: el notebook anterior (`06_senas_propias`) selecciono el mejor
modelo de los tres construidos en este laboratorio (`random_forest_aug`, 0.9709 de accuracy en
`x_test`) y lo puso a prueba con 15 fotos reales, tomadas con celular, de tres integrantes del
grupo. El resultado fue un colapso de 0.9709 a 0.0000: el modelo no acerto ni una sola de las
15 fotos. Ese experimento ya es evidencia de que este dataset, tal como esta, no sirve para un
producto real de accesibilidad. Lo que sigue es explicar por que y que se necesitaria para
cerrar esa brecha.

## **Limitaciones del dataset**

**Fondo e iluminacion.** El EDA (`01_eda`) ya señalaba que dentro de cada clase el fondo y la
iluminacion son consistentes, señal de pocas sesiones de grabacion en vez de miles de personas
distintas grabando en condiciones distintas. `06_senas_propias` confirmo el efecto practico de
eso: al cambiar el fondo de estudio por el fondo de un cuarto cualquiera, el modelo que
acertaba 97 de cada 100 imagenes de Kaggle dejo de reconocer una sola letra, e incluso predijo
`nothing` (ninguna mano visible) en 10 de los 15 casos. El modelo no aprendio unicamente la
forma de la mano, aprendio tambien el entorno en el que esa mano fue fotografiada.

**Angulo de camara y distancia.** Las 87,000 imagenes del dataset comparten encuadre: mano
centrada, misma distancia aproximada a la camara, mismo angulo. Una app real se usa con el
celular en la mano de quien firma, a la distancia y angulo que le resulten comodos, lo cual
varia de una persona a otra y de una sesion a otra. El dataset no tiene forma de enseñarle al
modelo que esa variacion no cambia la letra.

**Tamaño de mano.** No hay variacion de tamaño de mano en el dataset mas alla de la que aporta
una sola persona (o un numero reducido de personas) por sesion de grabacion. Manos de niños,
manos mas grandes o mas pequeñas, o con proporciones distintas, no estan representadas, y el
tamaño de mano interactua con la distancia a la camara, lo que multiplica la variabilidad no
cubierta.

**Tono de piel.** Este es probablemente el sesgo mas serio, aunque el mas dificil de cuantificar
con las herramientas de este laboratorio: si el dataset se grabo en pocas sesiones, es
razonable asumir que tambien esta grabado con muy pocos tonos de piel distintos, quizas uno
solo. Un modelo entrenado asi puede terminar aprendiendo a asociar cierto rango de intensidad de
pixel con "esto es una mano", y fallar de forma sistematica con usuarios cuyo tono de piel cae
fuera de ese rango. Esto no es un problema de accuracy general, es un problema de a quien
excluye el sistema, que es exactamente lo que un producto de accesibilidad no se puede permitir.

## **El caso especifico del Random Forest**

El modelo seleccionado en `06_senas_propias`, `random_forest_aug`, es tambien el que mejor
ilustra el problema: tenia el mejor accuracy de los tres en `x_test` (0.9709) y, al mismo
tiempo, colapso por completo con las fotos reales, con 10 de las 15 predicciones directamente
en `nothing`. La explicacion tiene que ver con como funciona el algoritmo, no con mala suerte.

Un Random Forest sobre pixeles aplanados (`aplanar` en `src/modelado.py`, de 64x64x1 a 4096
caracteristicas) construye sus arboles cortando sobre el valor de pixeles en **posiciones
fijas** de la imagen. El pixel numero 2000, por ejemplo, siempre corresponde a la misma
coordenada (x, y) del encuadre. Ese enfoque funciona bien cuando el encuadre es consistente,
como en el dataset de Kaggle, porque esa posicion fija realmente cae siempre sobre la mano o
siempre sobre el fondo. Pero en cuanto la mano se desplaza dentro del encuadre, cambia de
tamaño o el fondo deja de ser uniforme, esa posicion fija deja de significar lo mismo de una
foto a otra. El arbol no tiene ningun mecanismo de invarianza a traslacion o escala, a
diferencia de una CNN, en donde las convoluciones y el pooling permiten reconocer un patron sin
importar exactamente en que parte de la imagen aparece. Al evaluar las tres variantes campeonas contra las mismas 15 fotografias, ninguna generaliza
fuera del dataset: el Random Forest acierta 0 de 15, la red simple 0 de 15 y la CNN apenas 1
de 15. No obstante, la forma en que fallan si las distingue. El Random Forest colapsa hacia
`nothing` en 11 de los 15 casos y la red simple en 3, mientras que la CNN no predice `nothing`
ni una sola vez. Es decir, el Random Forest no solo se equivoca de letra, sino que directamente
deja de reconocer que hay una mano en la imagen, que es justamente lo que predice su falta de
invarianza a traslacion y escala: al cambiar el fondo y la posicion, la combinacion de pixeles
en posiciones fijas se parece mas a la clase "sin mano visible" que a cualquier letra. Bajo
esta idea, gran parte de lo que las tres aprendieron a clasificar no era la forma de la letra,
sino la posicion y el fondo en el que esa letra fue fotografiada en el dataset.

**Recomendacion concreta para el Random Forest.** En vez de alimentarlo con pixeles crudos
aplanados, extraer primero una region de interes centrada y normalizada en tamaño alrededor de
la mano (por ejemplo, usando un detector de manos como MediaPipe Hands para obtener el bounding
box y los landmarks) y calcular caracteristicas sobre esa region ya alineada: histogramas de
gradiente orientado (HOG) o directamente las coordenadas normalizadas de los landmarks de la
mano. Esto traslada la invarianza a posicion y escala del modelo (que el Random Forest no
tiene) al paso de preprocesamiento, y ademas elimina el fondo del problema por completo, ya que
el modelo nunca ve pixeles fuera de la mano. Esta misma idea, separar "encontrar la mano" de
"clasificar la letra", tambien beneficiaria a la CNN y a la red simple si se usaran en
produccion, ya que ninguna de las tres familias vio nunca un fondo distinto al del estudio de
grabacion durante el entrenamiento.

## **Que se necesitaria para un caso de uso real como SignBridge**

1. **Recoleccion de datos representativa.** Grabar el dataset propio con muchas personas
   distintas (no solo pocas sesiones repetidas 3000 veces), cubriendo variedad de tono de piel,
   tamaño de mano y, dentro de lo posible, alguna condicion que afecte la motricidad de la mano,
   ya que una parte real de los usuarios de un producto de accesibilidad tiene necesidades
   motrices distintas a las de quienes grabaron el dataset original.

2. **Deteccion de mano como paso previo a la clasificacion.** Como se explico arriba, un
   detector de manos que recorte y normalize la region antes de clasificar resuelve de raiz el
   problema de fondo/encuadre que `06_senas_propias` expuso con el modelo seleccionado, y que
   por diseño afecta a cualquier modelo entrenado sobre la imagen completa, no solo al Random
   Forest.

3. **Captura en condiciones variadas durante el entrenamiento.** Grabar (no solo aumentar
   sinteticamente) con distintos fondos, distancias de camara y condiciones de luz, para que la
   variacion que un usuario real introduce este representada en los datos desde el origen y no
   dependa de que la rotacion/zoom/brillo sinteticos de `03_augmentation` la cubran por
   aproximacion.

4. **Metrica de evaluacion por subgrupo.** Medir accuracy no solo de forma agregada sino
   desglosada por tono de piel, tamaño de mano y dispositivo de captura, para poder detectar si
   el modelo excluye sistematicamente a algun grupo antes de llevarlo a produccion, en lugar de
   confiar en un unico numero de accuracy general que puede esconder ese tipo de sesgo.